In [1]:
import  os
import fitz  # PyMuPDF
# from langgraph_utils.Info_extractor import InfoExtractorAgent
# from langgraph_utils.chat_history import SnowflakeChatMessageHistory
import json
# from langgraph_utils.file_parser import FileParser
# from langgraph_utils.digitization import main_handler
import uuid
# from langgraph_utils import creds
# from langgraph_utils.variables import PROJECT_NAME, SECRET_NAME, TOKEN_KEY
from utils.connection import get_dataiku_client_and_project
import logging
import tempfile
import re
import pymupdf4llm
from dataikuapi import DSSClient
from dataikuapi.dss.project import DSSProject
import pandas as pd
import asyncio

from soa_extraction.opensearch_utils import OpensearchUtil, create_embeddings_new
import nest_asyncio



# import from GLOBAL SHARED CODE
from utils import connection 
# imports from library 
# from utilities.creds import RD_PROJECT_NAME
# from variables import SECRET_NAME , TOKEN_KEY
# from utilities.logging_config import logging
from IPython.core.display import display, HTML
from uuid import uuid4

/tmp/ipykernel_435327/646784850.py:32: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [2]:
import dataikuapi
DATAIKU_HOST = "http://10.45.152.66:10000"
API_SECRET_KEY = "dkuaps-b3EsRXVjU3w4y7nd4KEwibEr04CjFPZr"          
PROJECT_NAME = "ECSGENERATION"   

# client, proj = get_dataiku_client_and_project(PROJECT_NAME, SECRET_NAME, TOKEN_KEY)
client = dataikuapi.DSSClient(DATAIKU_HOST, API_SECRET_KEY)
proj = client.get_project(PROJECT_NAME)

In [3]:
class readExcel:
    def __init__(self,client,proj):
        self.proj = proj
        self.client = client
        self.s3_folder_dataset_id = proj.get_variables()['local'].get('ecs_excel') # change file upload 
        self.input_folder = proj.get_managed_folder(self.s3_folder_dataset_id)
        self.files = self.input_folder.list_contents()["items"]
        self.toc_page_limit = 20
        self.config = proj.get_variables()["local"]
        print(self.files)
        
    def standard_ecs_excel_read(self,path= ''):
        """
        This Class Function is responsible for reading the Standard Excel Files
        """
        try:
            result = []

            for file in self.files:
                path = file['path']
                print(path)
                parts = path.strip('/').split('/')
                if file_path == '':
                    if "Standard" in path:

                        result.append({

                            "path": path,


                        })
                else:
                      path = file_path  
                    
                
            df_list = []
            for file in result:
                if file_path == '':
                    with self.input_folder.get_file(file['path']) as stream:
                        file_bytes = stream.raw.data
                    #     df = pd.read_excel(file_bytes, sheet_name='SV Domain Validations')  
                        sheet_names = pd.ExcelFile(file_bytes).sheet_names
                        
                combined_df = pd.DataFrame()
                for sheet in sheet_names:
                    if 'Rave' in sheet:
                        print('do not read the excel')
                        continue
                    with self.input_folder.get_file(file['path']) as stream:
                        file_bytes = stream.raw.data
                        dict_df = pd.read_excel(file_bytes, sheet_name=sheet)
                #     print(type(dict_df))    
                #     temp_df = pd.DataFrame([dict_df])
                    temp_df = dict_df
                    combined_df = pd.concat([combined_df,temp_df],ignore_index=True)
                df_list.append(combined_df)
                
            return df_list
        except:
            import traceback
            t = traceback.format_exc()
            raise f"error caused due to {t}"

In [4]:
ecs_obj = readExcel(client,proj)


[{'path': '/Histoical/384-201-00004_Database Check Guide_04MAR2025.xlsx', 'size': 154645, 'lastModified': 1758714856000}, {'path': '/Histoical/MAC186_X11-201-00001_Data Validation Plan_V6.0 15Aug2025.xlsx', 'size': 168923, 'lastModified': 1758714856000}, {'path': '/Histoical/Otsuka 384-201-00002_Data Validation Specification_V4.0_29Apr2025.xlsx', 'size': 1104146, 'lastModified': 1758714838000}, {'path': '/Standard/Copy of Otsuka Standard Edit Check Specifications (1).xlsx', 'size': 131156, 'lastModified': 1757487453000}]


In [5]:
ecs_list = ecs_obj.standard_ecs_excel_read()

/Histoical/384-201-00004_Database Check Guide_04MAR2025.xlsx


TypeError: exceptions must derive from BaseException

In [29]:
name_uuid_map = {}
name_ids = []
row_ids = []
import uuid
for index, row in ecs_list[0].iterrows():
    name = row["OGCMS Data Collection Domain Name"]
    if name not in name_uuid_map:
        name_uuid_map[name] = str(uuid.uuid4())
    name_ids.append(name_uuid_map[name])
    
    # Assign new UUID for each row
    row_ids.append(str(uuid.uuid4()))

ecs_list[0]['form_name_id'] = name_ids
ecs_list[0]['row_id'] = row_ids
ecs_list[0]

,VALIDATION ID,OGCMS VERSION,OGCMS Data Collection Domain Name,OGCMS Data Collection Domain CRF,OGCMS Data Collection Variable Text,OGCMS Data Collection Variable Name,VALIDATION LOGIC,REASONING,ACTION,ACTION DETAILS,form_name_id,row_id
0,MVAL_SV001,OGCMS v2,Subject Visits,SV,"If no, what was the reason the visit was not d...",SVREASOC,"(SV.SVOCCUR == ""No"") then (SV.SVREASOC is ente...",If Visit did not occur then reason visit not d...,SV.SVREASOC is enterable,<n/a>,3c36705a-c794-48a0-aa54-0453ea51bde5,b7bb918d-e7a5-474d-a43b-b0714a4d4ab4
1,MVAL_SV002,OGCMS v2,Subject Visits,SV,Visit Date,VISDAT,"(SV.SVOCCUR == ""Yes"") then (SV.VISDAT is enter...",If Visit did occur then visit date is a valid ...,SV.VISDAT is enterable,<n/a>,3c36705a-c794-48a0-aa54-0453ea51bde5,2fafa68d-cbe5-444a-86ec-7da8efc1fcea
2,MVAL_SV003,OGCMS v2,Subject Visits,SV,Visit Time,VISTIM,"(SV.SVOCCUR == ""Yes"") then (SV.VISTIM is enter...",If Visit did occur then visit date is a valid ...,SV.VISTIM is enterable,<n/a>,3c36705a-c794-48a0-aa54-0453ea51bde5,70447c2b-519d-48b6-bc7f-8bd62ce418ae
3,MVAL_SV004,OGCMS v2,Subject Visits,SV,What method was used to conduct the visit?,SVCNTMOD,"(SV.SVOCCUR == ""Yes"") then (SV.SVCNTMOD is ent...","If Visit did occur then ""What method was used ...",SV.SVCNTMOD is enterable,<n/a>,3c36705a-c794-48a0-aa54-0453ea51bde5,81c2490b-35de-4039-a623-40315eb18930
4,MVAL_SV005,OGCMS v2,Subject Visits,SV,Visit,VISIT,(SV.VISIT is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,3c36705a-c794-48a0-aa54-0453ea51bde5,051006ae-f529-47f7-904e-0f275e97738a
...,...,...,...,...,...,...,...,...,...,...,...,...
281,MVAL_DS_FU009,OGCMS v2,\nPost-treatment Follow-up,DS_FU,Date of contact/Date of final contact attempt,DSSTDAT,(DS_FU.DSSTDAT is an invalid date),Field must not be an invalid date such as 31Fe...,prompt user with ACTION DETAILS,<query the field for invalid date>,5005b2af-8f33-40e7-a3e0-1463f38c1b69,94940a0b-0e88-451a-846d-c507f6f0cc19
282,MVAL_DS_FU010,OGCMS v2,\nPost-treatment Follow-up,DS_FU,Contact Method,QVAL_DSCONTCT,(DS_FU.QVAL_DSCONTCT is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,5005b2af-8f33-40e7-a3e0-1463f38c1b69,b67dd4ae-7afb-4459-b9aa-33aeccdfea20
283,MVAL_DS_FU011,OGCMS v2,\nPost-treatment Follow-up,DS_FU,Specify other contact,QVAL_DSCONTOT,(DS_FU.QVAL_DSCONTOT is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,5005b2af-8f33-40e7-a3e0-1463f38c1b69,ab0424d5-8914-4ade-85a0-1de7287d05b3
284,MVAL_DS_FU012,OGCMS v2,Post-treatment Follow-up,DS_FU,Date of Contact/Final Contact Attempt,DSSTDAT,(DS_FU.DSSTDAT < DS_COMP_EOT.DSSTDAT),no post treatment contact date should ever be ...,prompt user with ACTION DETAILS,No post treatment contact date should be prior...,8b6915a3-7c5f-470d-bb19-2d2381a1c112,47780e71-9ba6-40a0-a31d-92beb2e61ac4


In [46]:
form_name_list = []
field_value_list = []
mix_list = []
for index, row in ecs_list[0].iterrows():
    form_name_list.append(row['OGCMS Data Collection Domain Name'])
    field_value_list.append(row['OGCMS Data Collection Variable Text'])
    mix_list.append(f"{row['OGCMS Data Collection Domain Name']} : {row['OGCMS Data Collection Variable Text']}")

In [47]:
nest_asyncio.apply()

form_embeddings =  asyncio.run(create_embeddings_new(proj, form_name_list, proj.get_variables()['local'].get('default_embeddings_model_id')))
fields_embeddings =  asyncio.run(create_embeddings_new(proj, field_value_list, proj.get_variables()['local'].get('default_embeddings_model_id')))
mix_embeddings =  asyncio.run(create_embeddings_new(proj, mix_list, proj.get_variables()['local'].get('default_embeddings_model_id')))




In [30]:
len(form_embeddings['response'])
ecs_list[0].columns

Index(['VALIDATION ID', 'OGCMS VERSION', 'OGCMS Data Collection Domain Name',
       'OGCMS Data Collection Domain CRF',
       'OGCMS Data Collection Variable Text',
       'OGCMS Data Collection Variable Name', 'VALIDATION LOGIC', 'REASONING',
       'ACTION', 'ACTION DETAILS', 'form_name_id', 'row_id'],
      dtype='object')

In [49]:
generic_mapping = []
for (index,row) , form_emb,field_emb,mix_emb in zip(ecs_list[0].iterrows()
                                                    ,form_embeddings['response'],
                                                    fields_embeddings['response'],
                                                   mix_embeddings['response']):
    generic_mapping.append(
        {
            "ecs_id": row['row_id'],
            "form_id" : row['form_name_id'],
            "validation_id": row['VALIDATION ID'],
            "form_name": row['OGCMS Data Collection Domain Name'],
            "form_domain_name":row['OGCMS Data Collection Domain CRF'],
            "form_field_value": row['OGCMS Data Collection Variable Text'],
            "variable_name":row['OGCMS Data Collection Variable Name'],
            "validation_logic":row['VALIDATION LOGIC'],
            "reasoning":row['REASONING'],
            "action": row['ACTION'],
            "action_details": row['ACTION DETAILS'],
            "source": "Standard",
            "path":"/Standard/Copy of Otsuka Standard Edit Check Specifications (1).xlsx",
            "form_name_vector":form_emb,
            "form_field_value_vector":field_emb,
            "form_field_vector":mix_emb
            
        }
    )

In [50]:
generic_mapping[28]

{'ecs_id': '40f6e22a-7885-46b4-8319-0d6d8295d0f0',
 'form_id': '4a5552de-4909-4846-bd25-3c28f9351b53',
 'validation_id': 'MVAL_DM003',
 'form_name': 'Demographics',
 'form_domain_name': 'DM',
 'form_field_value': 'If Other, specify race',
 'variable_name': 'QVAL_RACEOTH',
 'validation_logic': '(DM.RACE_OTHER is marked) then (DM.QVAL_RACEOTH is enterable)',
 'reasoning': 'If Race Other is marked then "If Other, specify race" is a valid field for entry',
 'action': 'DM.QVAL_RACEOTH is enterable',
 'action_details': '<n/a>',
 'source': 'Standard',
 'path': '/Standard/Copy of Otsuka Standard Edit Check Specifications (1).xlsx',
 'form_name_vector': [0.005614279769361019,
  0.014669911935925484,
  0.04412280023097992,
  0.05616400018334389,
  0.057605549693107605,
  -0.00778720760717988,
  -0.038328323513269424,
  0.020012134686112404,
  -0.01243691984564066,
  0.019079364836215973,
  0.011871605180203915,
  -0.0012516416609287262,
  -0.039148032665252686,
  -0.005448218900710344,
  0.02794

In [51]:
import json
from opensearchpy.helpers import bulk
from opensearchpy import OpenSearch
opensearch_client = OpensearchUtil(client, proj)

def bulk_insert( index_name, documents):
    project = proj
        
    # Retrieve credentials from Opensearch connection 
    project_configs = project.get_variables()["local"]
    opensearch_connection = project_configs["opensearch_connection"]



    conn_info = client.get_connection(opensearch_connection).get_info()

    opensearchclient = OpenSearch(
            hosts=[{"host": conn_info["params"]["host"], "port": conn_info["params"]["port"]}],
            http_auth = (conn_info["params"]["username"], conn_info["params"]["password"]),
            use_ssl=conn_info["params"]["ssl"],
            verify_certs=False
    )
    

    docs_to_store = []
    for document in documents:

        action = {
            "_op_type": "index",  # Operation type (index = insert)
            "_index": index_name,  # Index name
            "_id": document['ecs_id'],  # Use the unique document ID
            "_source": document  # Document body
        }
        docs_to_store.append(action)

    success, failed = bulk(opensearchclient, docs_to_store)
    print(f"Successfullly indexed {success} documents.")
    print(f"Failed to index {failed} documents.")
    if failed:
        raise Exception("Document indexing encountered an unknown error. ")


# index_name = proj.get_variables()['local'].get('ecs_opensearch')
index_name = "${projectKey}_test_ecs"
dataiku_project_var = "${projectKey}"
if dataiku_project_var in index_name:
    index_name = index_name.replace(dataiku_project_var, opensearch_client.project.project_key).lower()
    print("index_name",index_name)
bulk_insert(index_name, generic_mapping)

{'type': 'ElasticSearch', 'params': {'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'username': 'genai-admin', 'password': 'qPk7Jf5vcyXDS!**332gXTvSfmcauvr9', 'port': 443, 'ssl': True, 'trustAnySSLCertificate': True, 'dialect': 'ES_7', 'dkuProperties': [], 'namingRule': {'indexNameDatasetNamePrefix': '${projectKey}_'}, 'authType': 'PASSWORD', 'oauth': {'refreshTokenRotation': False}, 'aws': {'service': 'OPENSEARCH_SERVERLESS', 'credentialsMode': 'KEYPAIR', 'customAWSCredentialsProviderParams': []}}, 'credentialsMode': 'GLOBAL', 'proxySettingsAsString': ''}
opensearch <OpenSearch([{'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'port': 443}])>
index_name ecsgeneration_test_ecs


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/opensearchpy/connection/http_urllib3.py:214: UserWarning: Connecting to https://aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com:443 using SSL with verify_certs=False is insecure.
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


Successfullly indexed 286 documents.
Failed to index [] documents.


In [6]:
#necessary imports 
from datetime import datetime
import traceback
import dataikuapi
import io

import base64
import uuid
# import from GLOBAL SHARED CODE
from utils import connection 
# imports from library 
from utilities.variables import RD_PROJECT_NAME
from utilities.variables import SECRET_NAME , TOKEN_KEY
from utilities.logging_config import logging
import os
import traceback
import subprocess
import tempfile
import boto3

def file_upload(user_id,file_b64,file_name):
    """
    The following function is responsible for taking base64 string of the file and upload it to the desired managed folder .
    
    Parameters:
        user_id : str
        file_b64 : str
        file_name : str
        
    Return:
        result : dict 
    """
    try:
        logging.info("Intializing client for file_upload function ")
        
        # setting up the connections and table name from local variables 
        #client = dataikuapi.DSSClient(DATAIKU_HOST, API_SECRET_KEY)
        #project = client.get_project(RD_PROJECT_NAME)
        DATAIKU_HOST , API_SECRET_KEY  = connection.get_dataiku_host_and_api_key(RD_PROJECT_NAME,SECRET_NAME,TOKEN_KEY)
        
        client = dataikuapi.DSSClient(DATAIKU_HOST, API_SECRET_KEY)
        project = client.get_project(RD_PROJECT_NAME)
        proj_vars = project.get_variables()["local"]
        snowflake_conn = proj_vars.get("snowflake_connection_string")
        
        
        input_folder =  project.get_managed_folder(proj_vars.get("ecs").get("file_upload"))
        table_file_upload = proj_vars.get("ecs", {}).get("ecs_file_upload", {})
        logging.info(f"Fetching the table name from the local variables ")
        
        protocol_id = str(uuid.uuid4())
        logging.info(f"Generating protocol id : {protocol_id}")
        
        
        
        file_path = f"/{file_name}"
        
        

        
           
        file = io.BytesIO(base64.b64decode(file_b64))
        file.seek(0)
        input_folder.put_file(f"{file_path}", file)
        
            
        logging.info("Upload file to to managed folder ")
        
        if "docx" in file_path:
            
            try:
                
                temp_dir = tempfile.mkdtemp(prefix="docx2pdf_")
                print(f"Using temporary directory: {temp_dir}")

                # Determine input path
                input_docx_path =  file_path
                print(f"Fetching DOCX file from managed folder: {input_docx_path}")

                # Get managed folder
                folder_id = project.get_variables()['local'].get("crf").get("crf_temp_upload_folder")
                folder = project.get_managed_folder(folder_id)

                # Download DOCX from managed folder → local temp
                local_docx_path = os.path.join(temp_dir, "temp_doc.docx")
                with folder.get_file(input_docx_path) as stream:
                    with open(local_docx_path, "wb") as f:
                        f.write(stream.raw.data)
                print(f"Downloaded DOCX to temporary path: {local_docx_path}")

                # Convert DOCX → PDF using LibreOffice headless
                subprocess.run([
                    "libreoffice", "--headless", "--convert-to", "pdf", "--outdir", temp_dir, local_docx_path
                ], check=True)

                local_pdf_path = os.path.join(temp_dir, "temp_doc.pdf")
                if not os.path.exists(local_pdf_path):
                    raise FileNotFoundError("LibreOffice did not produce a PDF")

                print("Conversion successful.")

                # Upload PDF back to managed folder
                output_pdf_path = file_path.replace('.docx', '.pdf')
                with open(local_pdf_path, "rb") as pdf_file:
                    pdf_stream = io.BytesIO(pdf_file.read())

                #out_folder_id = project.get_variables()['local'].get('crf_temp_upload_folder')
                output_folder = project.get_managed_folder(folder_id)
                output_folder.put_file(output_pdf_path, pdf_stream)
                print(f"Uploaded PDF to managed folder: {output_pdf_path}")

                file_path = output_pdf_path
                
            except Exception as e:
                import traceback
                error = traceback.format_exc()
                print(f"Error encountered: {error}")
                return {"content": f"Error: {error}"}

        
        percent = 0
        insert_query  = f"""INSERT INTO {table_file_upload} (
                              "user_id",
                              "crf_file_id",
                              "file_path",
                              "file_name",
                              "is_digitized",
                              "uploaded_at",
                              "digitization_percent"
                            ) VALUES (
                              '{user_id}',
                              '{protocol_id}',
                              '{file_path}',
                              '{file_name}',
                              False,
                              current_timestamp,
                              {percent}
                            );"""
        logging.info(f"Running query ")
        client.sql_query(query = insert_query,connection = snowflake_conn,  post_queries=["COMMIT"])
        logging.info(f"sql query : {insert_query} ran successfully ")
        print(f"sql query : {insert_query} ran successfully ")
        
        return {"message":"success","crf_file_id":protocol_id,"file_path":file_path}
    except Exception as e:
        t = traceback.format_exc()
        logging.error(f"Error caused due to {t}")
        return {"message":f"error caused due to {t}"}

In [9]:
file_upload('1','JVBERi0xLjQKMSAwIG9iago8PC9UeXBlIC9DYXRhbG9nCi9QYWdlcyAyIDAgUgo+PgplbmRvYmoK MiAwIG9iago8PC9UeXBlIC9QYWdlcwovS2lkcyBbMyAwIFJdCi9Db3VudCAxCj4+CmVuZG9iagoz IDAgb2JqCjw8L1R5cGUgL1BhZ2UKL1BhcmVudCAyIDAgUgovTWVkaWFCb3ggWzAgMCA1OTUgODQy XQovQ29udGVudHMgNSAwIFIKL1Jlc291cmNlcyA8PC9Qcm9jU2V0IFsvUERGIC9UZXh0XQovRm9u dCA8PC9GMSA0IDAgUj4+Cj4+Cj4+CmVuZG9iago0IDAgb2JqCjw8L1R5cGUgL0ZvbnQKL1N1YnR5 cGUgL1R5cGUxCi9OYW1lIC9GMQovQmFzZUZvbnQgL0hlbHZldGljYQovRW5jb2RpbmcgL01hY1Jv bWFuRW5jb2RpbmcKPj4KZW5kb2JqCjUgMCBvYmoKPDwvTGVuZ3RoIDUzCj4+CnN0cmVhbQpCVAov RjEgMjAgVGYKMjIwIDQwMCBUZAooRHVtbXkgUERGKSBUagpFVAplbmRzdHJlYW0KZW5kb2JqCnhy ZWYKMCA2CjAwMDAwMDAwMDAgNjU1MzUgZgowMDAwMDAwMDA5IDAwMDAwIG4KMDAwMDAwMDA2MyAw MDAwMCBuCjAwMDAwMDAxMjQgMDAwMDAgbgowMDAwMDAwMjc3IDAwMDAwIG4KMDAwMDAwMDM5MiAw MDAwMCBuCnRyYWlsZXIKPDwvU2l6ZSA2Ci9Sb290IDEgMCBSCj4+CnN0YXJ0eHJlZgo0OTUKJSVF T0YK','3')

sql query : INSERT INTO SK_AIWRITER_CRF.CTL_CRF_FILE_UPLOAD (
                              "user_id",
                              "crf_file_id",
                              "file_path",
                              "file_name",
                              "is_digitized",
                              "uploaded_at",
                              "digitization_percent"
                            ) VALUES (
                              '1',
                              '8dce8d8e-5c7f-40cd-bc28-7f9b559f9571',
                              '/3',
                              '3',
                              False,
                              current_timestamp,
                              0
                            ); ran successfully 


{'message': 'success',
 'crf_file_id': '8dce8d8e-5c7f-40cd-bc28-7f9b559f9571',
 'file_path': '/3'}

In [3]:
#necessary imports 
from datetime import datetime
import traceback
import dataikuapi
import io

import base64
import uuid
# import from GLOBAL SHARED CODE
from utils import connection 
# imports from library 
import re
from utilities.variables import RD_PROJECT_NAME
from utilities.variables import SECRET_NAME , TOKEN_KEY
from utilities.logging_config import logging
from soa_extraction.crf_extraction import HistoricalCRF
import traceback
import os


def extract_crf(file_path):
    """
    Input Args:
        file_path: str
        
    Response:
        result : dict
    """
    try:
        logging.info("Intializing client for file_upload function ")
       
        DATAIKU_HOST , API_SECRET_KEY  = connection.get_dataiku_host_and_api_key(RD_PROJECT_NAME,SECRET_NAME,TOKEN_KEY)
        
        client = dataikuapi.DSSClient(DATAIKU_HOST, API_SECRET_KEY)
        project = client.get_project(RD_PROJECT_NAME)
        proj_vars = project.get_variables()["local"]
        
        snowflake_conn = proj_vars.get("snowflake_connection_string")
        
        obj = HistoricalCRF(client , project)
        final_list = []
        response = obj.historical_mapping(file_path)

        for resp in response:
            form_name = resp['source_data']['assessments']
            match = re.search(r'Form[:\s]*(.*)', form_name)
            if match:
                form_name = match.group(1)
            for field in resp['source_data']['fields']:
                field_name = field['field_name']
                final_list.append({
                    "form_name": form_name,
                    "field_name": field_name
                })

        # ✅ Deduplicate based on both form_name and field_name
        seen = set()
        deduped_list = []
        for item in final_list:
            key = (item["form_name"].strip().lower(), item["field_name"].strip().lower())
            if key not in seen:
                seen.add(key)
                deduped_list.append(item)

        return {
            "result": deduped_list
        }

    except:
        t = traceback.format_exc()
        logging.error(f"Error caused due to {t}")
        return {"message": f"Error caused due to {t}"}


In [4]:
file_path= "/Annotated_Otsuka_405 201 00157_00150405_v1.0_Complete eCRF (1).pdf"
out = extract_crf(file_path)

In [6]:
'Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit'
len(out['result'])

895